# Redis & Celery

## Redis Data Structures

Redis is an in-memory key-value store that supports multiple data types:

| Type | Use Case |
|------|----------|
| String | Simple values, JSON blobs |
| List | Queues (FIFO/LIFO) |
| Set | Unique collections |
| Hash | Objects with key-value pairs |
| Sorted Set | Ranked lists (leaderboards) |

In Django caching, strings are used most often. For queues and counters, choose the appropriate type.


## Installing Redis

**Ubuntu / WSL:**
```bash
sudo apt update && sudo apt install redis-server
sudo systemctl enable redis-server && sudo systemctl start redis-server
```

**macOS:**
```bash
brew install redis && brew services start redis
```

**Test connection:**
```bash
redis-cli ping   # → PONG
```


## Connecting Redis to Django as a Cache Backend

```bash
pip install django-redis redis
```

```python
# settings.py
CACHES = {
    "default": {
        "BACKEND": "django_redis.cache.RedisCache",
        "LOCATION": "redis://127.0.0.1:6379/1",
        "OPTIONS": {
            "CLIENT_CLASS": "django_redis.client.DefaultClient",
            "IGNORE_EXCEPTIONS": True,
        },
        "TIMEOUT": 60,
        "KEY_PREFIX": "myapp",
    }
}
```


## Redis CLI Commands

| Command | Description | Example Output |
|---------|-------------|----------------|
| `PING` | Test connection | `PONG` |
| `SELECT n` | Switch to DB n | (none) |
| `KEYS pattern` | List matching keys | `1) "myapp:bestsellers:v1"` |
| `TTL key` | Seconds remaining | `54` |
| `DEL key` | Delete a key | `1` |
| `FLUSHDB` | Clear current DB | `OK` |
| `TYPE key` | Show data type | `string` |
| `SET k v` / `GET k` | Store/retrieve | `OK` / value |

```bash
redis-cli -n 1       # connect to DB 1
KEYS *bestsellers*
TTL "myapp:bestsellers:v1"
```


## Sync vs Async: The Problem

When a view sends an email or generates a report synchronously, the HTTP response is blocked until the task completes:

```python
@api_view(["POST"])
def create_order(request):
    order = Order.objects.create(email=request.data["email"])
    # This blocks the response for 3 seconds
    import time; time.sleep(3)
    send_mail(subject="Order Confirmation", message="Thanks!", ...)
    return Response({"order_id": order.id})
```

For slow tasks, offload them to a background worker using Celery.


## Celery Setup

Celery is a distributed task queue. Redis acts as the broker (the queue).

```bash
pip install celery redis
```

```python
# myproject/celery.py
import os
from celery import Celery

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "myproject.settings")
app = Celery("myproject")
app.config_from_object("django.conf:settings", namespace="CELERY")
app.autodiscover_tasks()
```

```python
# settings.py
CELERY_BROKER_URL = "redis://127.0.0.1:6379/0"
CELERY_RESULT_BACKEND = "redis://127.0.0.1:6379/2"
CELERY_ACCEPT_CONTENT = ["json"]
CELERY_TASK_SERIALIZER = "json"
CELERY_RESULT_SERIALIZER = "json"
```

Connect Celery to Django in `__init__.py`:
```python
# myproject/__init__.py
from .celery import app as celery_app
__all__ = ("celery_app",)
```


## Defining and Calling Tasks

```python
# orders/tasks.py
from celery import shared_task
from django.core.mail import send_mail

@shared_task
def send_email_async(order_id):
    from .models import Order
    order = Order.objects.get(id=order_id)
    send_mail(
        subject=f"Order #{order.id} Confirmation",
        message=f"Thanks for your order, {order.email}!",
        from_email="noreply@example.com",
        recipient_list=[order.email],
    )
    return f"Email sent for order {order_id}"
```

Call the task asynchronously:
```python
@api_view(["POST"])
def create_order(request):
    order = Order.objects.create(email=request.data["email"])
    send_email_async.delay(order.id)   # queues the task; returns immediately
    return Response({"order_id": order.id})
```

Start the worker:
```bash
celery -A myproject worker -l INFO --concurrency=4
```


## Result Backend

The result backend stores task status and return values:

```python
result = send_email_async.delay(order_id)
print(result.id)      # UUID of the task
print(result.status)  # PENDING, STARTED, SUCCESS, FAILURE
print(result.get())   # block until done, then return the value
```


## Celery Beat: Periodic Tasks

Celery Beat schedules tasks to run automatically on a cron schedule.

```bash
pip install django-celery-beat
python manage.py migrate
```

```python
# settings.py
INSTALLED_APPS += ["django_celery_beat"]
```

Define a schedule in code:
```python
from celery.schedules import crontab

app.conf.beat_schedule = {
    "daily-sales-report": {
        "task": "orders.tasks.generate_sales_report",
        "schedule": crontab(minute=5, hour=0),   # every day at 00:05
        "args": ("daily",),
    },
}
```

Start Beat alongside the worker:
```bash
celery -A myproject beat -l INFO
```

Alternatively, manage schedules from the Django admin by creating Periodic Task entries.


## Summary

- Redis supports multiple data types (string, list, set, hash, sorted set) — choose the type that fits the use case.
- Celery decouples slow tasks from the HTTP request/response cycle.
- Redis acts as the broker; Celery workers pull jobs from it and execute them.
- Use `task.delay()` to queue a task asynchronously from a view.
- The result backend stores task status — useful for polling or chaining tasks.
- Celery Beat schedules periodic tasks using cron expressions, eliminating the need for OS-level cron jobs.
